# BTCUSDC Order-Flow Study: Burst Effect Visual Diagnostics

This notebook is for eyeballing calm-to-burst activity regimes before running return tests. It asks whether trade-sign imbalance is visually more meaningful when trades/sec or volume suddenly jumps relative to a prior local baseline.


## Load Data

Use the shared order-flow loader and trade-frame alignment helper. The analysis below starts from trade-aligned rows with timestamp, trade quantity, aggressor sign, and latest known top-of-book midprice.


In [ ]:
\
from pathlib import Path
import json
import sys
import uuid

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import HTML, display


from pathlib import Path
import importlib.util

def load_notebook_utils(start: Path | None = None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        utils_path = candidate / "notebooks" / "notebook_utils.py"
        if utils_path.is_file():
            spec = importlib.util.spec_from_file_location("notebook_utils", utils_path)
            if spec is None or spec.loader is None:
                raise ImportError(f"Could not load notebook utilities from {utils_path}")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            return module
    raise FileNotFoundError("Could not locate notebooks/notebook_utils.py")

notebook_utils = load_notebook_utils()
PROJECT_ROOT = notebook_utils.bootstrap_backtester_path()
find_backtester_root = notebook_utils.find_backtester_root
resolve_day_dir = notebook_utils.resolve_day_dir


## Calm-To-Burst Plot

This notebook uses clock-time windows.

Definitions:

- `burst window`: the current clock-time window used to compute current activity and current imbalance.
- `calm window`: the prior clock-time window used to compute the local baseline. The baseline is the rolling median of prior current-window activity values.
- `trades_per_sec_current`: number of trades in the current burst window divided by burst-window seconds.
- `volume_per_sec_current`: trade volume in the current burst window divided by burst-window seconds.
- `trades_per_sec_to_baseline`: `trades_per_sec_current / baseline_trades_per_sec`.
- `volume_to_baseline`: `volume_per_sec_current / baseline_volume_per_sec`.
- `signed_share`: `(buy_count - sell_count) / total_trade_count` inside the current burst window.
- `signed_share_ma`: a 240-second rolling mean of `signed_share`.
- `volume_imbalance`: `(buy_volume - sell_volume) / total_volume` inside the current burst window.
- `volume_imbalance_ma`: a 240-second rolling mean of `volume_imbalance`.

Editable controls:

- `burst min`: current activity window length in minutes.
- `calm min`: prior baseline window length in minutes.
- `burst mode`: which activity ratio must exceed the threshold.
  - `trades/sec`: only `trades_per_sec_to_baseline` is checked.
  - `volume`: only `volume_to_baseline` is checked.
  - `both`: both ratios must pass.
- `ratio th`: minimum activity-to-baseline ratio for a burst. Example: `3.0` means current activity is at least 3x the local baseline.
- `imb th`: absolute sign-imbalance cutoff used when `require imbalance` is enabled and for coloring directional bursts.
- `require imbalance`: if enabled, only windows with `abs(signed_share) >= imb th` are marked as bursts.
- `cooldown min`: minimum spacing between plotted burst markers. This reduces repeated markers from overlapping windows.

This first version intentionally has no separate calm-level threshold. A high ratio already means the current window is large relative to the prior local baseline. If that is too loose later, add a separate calm-level filter.


In [3]:
\
DEFAULT_BURST_MINUTES = 5.0
DEFAULT_CALM_MINUTES = 30.0
DEFAULT_RATIO_THRESHOLD = 3.0
DEFAULT_IMBALANCE_THRESHOLD = 0.50
DEFAULT_COOLDOWN_MINUTES = 5.0
GRID_FREQ = "1s"
SIGNED_SHARE_SMOOTH_WINDOW = "240s"
SIGNED_SHARE_SMOOTH_MIN_PERIODS = 30
VOLUME_IMBALANCE_SMOOTH_WINDOW = "240s"
VOLUME_IMBALANCE_SMOOTH_MIN_PERIODS = 30
MAX_PLOT_ROWS = 60_000


def _sample_evenly(frame: pd.DataFrame, *, max_rows: int) -> pd.DataFrame:
    if len(frame) <= max_rows:
        return frame.copy()
    step = int(np.ceil(len(frame) / max_rows))
    return frame.iloc[::step].copy()


def _cooldown_mask(ts: pd.Series, raw_mask: pd.Series, *, cooldown_minutes: float) -> pd.Series:
    selected = np.zeros(len(ts), dtype=bool)
    cooldown = pd.Timedelta(minutes=float(cooldown_minutes))
    last_selected = pd.Timestamp.min.tz_localize("UTC")
    for i, (timestamp, is_raw) in enumerate(zip(ts, raw_mask)):
        if bool(is_raw) and timestamp >= last_selected + cooldown:
            selected[i] = True
            last_selected = timestamp
    return pd.Series(selected, index=ts.index)


def build_clock_burst_features(
    trade_frame: pd.DataFrame,
    *,
    burst_minutes: float,
    calm_minutes: float,
) -> pd.DataFrame:
    burst_minutes = float(burst_minutes)
    calm_minutes = float(calm_minutes)
    if burst_minutes <= 0 or calm_minutes <= 0:
        raise ValueError("burst_minutes and calm_minutes must be positive")

    frame = trade_frame[["ts", "mid_at_book", "qty", "aggr_sign"]].copy().sort_values("ts")
    frame = frame.set_index("ts")
    frame["trade_count"] = 1.0
    frame["buy_count"] = (frame["aggr_sign"] > 0).astype(float)
    frame["sell_count"] = (frame["aggr_sign"] < 0).astype(float)
    frame["buy_qty"] = np.where(frame["aggr_sign"] > 0, frame["qty"], 0.0)
    frame["sell_qty"] = np.where(frame["aggr_sign"] < 0, frame["qty"], 0.0)

    bars = frame.resample(GRID_FREQ).agg(
        mid_at_book=("mid_at_book", "last"),
        trade_count=("trade_count", "sum"),
        buy_count=("buy_count", "sum"),
        sell_count=("sell_count", "sum"),
        total_volume=("qty", "sum"),
        buy_volume=("buy_qty", "sum"),
        sell_volume=("sell_qty", "sum"),
    )
    bars["mid_at_book"] = bars["mid_at_book"].ffill()

    burst_window = f"{int(round(burst_minutes * 60))}s"
    calm_window = f"{int(round(calm_minutes * 60))}s"
    burst_seconds = burst_minutes * 60.0

    out = pd.DataFrame(index=bars.index)
    out["mid_at_book"] = bars["mid_at_book"]
    out["trade_count_current"] = bars["trade_count"].rolling(burst_window, min_periods=1).sum()
    out["buy_count_current"] = bars["buy_count"].rolling(burst_window, min_periods=1).sum()
    out["sell_count_current"] = bars["sell_count"].rolling(burst_window, min_periods=1).sum()
    out["volume_current"] = bars["total_volume"].rolling(burst_window, min_periods=1).sum()
    out["buy_volume_current"] = bars["buy_volume"].rolling(burst_window, min_periods=1).sum()
    out["sell_volume_current"] = bars["sell_volume"].rolling(burst_window, min_periods=1).sum()

    out["trades_per_sec_current"] = out["trade_count_current"] / burst_seconds
    out["volume_per_sec_current"] = out["volume_current"] / burst_seconds
    out["signed_share"] = (out["buy_count_current"] - out["sell_count_current"]) / out["trade_count_current"].replace(0.0, np.nan)
    out["signed_share_ma"] = out["signed_share"].rolling(SIGNED_SHARE_SMOOTH_WINDOW, min_periods=SIGNED_SHARE_SMOOTH_MIN_PERIODS).mean()
    out["volume_imbalance"] = (out["buy_volume_current"] - out["sell_volume_current"]) / out["volume_current"].replace(0.0, np.nan)
    out["volume_imbalance_ma"] = out["volume_imbalance"].rolling(VOLUME_IMBALANCE_SMOOTH_WINDOW, min_periods=VOLUME_IMBALANCE_SMOOTH_MIN_PERIODS).mean()

    out["baseline_trades_per_sec"] = out["trades_per_sec_current"].shift(1).rolling(calm_window, min_periods=1).median()
    out["baseline_volume_per_sec"] = out["volume_per_sec_current"].shift(1).rolling(calm_window, min_periods=1).median()
    out["trades_per_sec_to_baseline"] = out["trades_per_sec_current"] / out["baseline_trades_per_sec"].replace(0.0, np.nan)
    out["volume_to_baseline"] = out["volume_per_sec_current"] / out["baseline_volume_per_sec"].replace(0.0, np.nan)
    return out.reset_index().dropna(subset=["mid_at_book", "signed_share", "volume_imbalance", "trades_per_sec_to_baseline", "volume_to_baseline"]).reset_index(drop=True)


def add_burst_flags(
    features: pd.DataFrame,
    *,
    burst_mode: str,
    ratio_threshold: float,
    imbalance_threshold: float,
    require_imbalance: bool,
    cooldown_minutes: float,
) -> pd.DataFrame:
    out = features.copy()
    rate_pass = out["trades_per_sec_to_baseline"] >= float(ratio_threshold)
    volume_pass = out["volume_to_baseline"] >= float(ratio_threshold)
    if burst_mode == "both":
        raw_burst = rate_pass & volume_pass
    elif burst_mode == "volume":
        raw_burst = volume_pass
    else:
        raw_burst = rate_pass

    if require_imbalance:
        raw_burst = raw_burst & (out["signed_share"].abs() >= float(imbalance_threshold))

    out["raw_burst"] = raw_burst
    out["burst_marker"] = _cooldown_mask(out["ts"], out["raw_burst"], cooldown_minutes=float(cooldown_minutes))
    out["burst_side"] = np.select(
        [out["signed_share"] >= float(imbalance_threshold), out["signed_share"] <= -float(imbalance_threshold)],
        [1, -1],
        default=0,
    ).astype(int)
    return out


def make_burst_figure(features: pd.DataFrame, *, imbalance_threshold: float, title: str) -> go.Figure:
    plot_df = _sample_evenly(features, max_rows=MAX_PLOT_ROWS)
    price_df = features[["ts", "mid_at_book"]].copy().sort_values("ts")
    burst_df = features.loc[features["burst_marker"]].copy()
    buy_bursts = burst_df.loc[burst_df["burst_side"] > 0]
    sell_bursts = burst_df.loc[burst_df["burst_side"] < 0]
    neutral_bursts = burst_df.loc[burst_df["burst_side"] == 0]

    fig = make_subplots(
        rows=7,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.34, 0.12, 0.12, 0.14, 0.10, 0.10, 0.08],
        vertical_spacing=0.065,
        subplot_titles=(
            "midprice with burst markers",
            "burst-window sign imbalance",
            "burst-window volume imbalance",
            "trades/sec: current and baseline",
            "trades/sec to baseline",
            "volume/sec to baseline",
            "burst-window volume",
        ),
    )
    fig.add_trace(go.Scattergl(x=price_df["ts"], y=price_df["mid_at_book"], mode="lines", line=dict(color="#111827", width=1), name="midprice", hoverinfo="skip"), row=1, col=1)
    for data, name, color, symbol in [
        (buy_bursts, "buy burst", "#16a34a", "triangle-up"),
        (sell_bursts, "sell burst", "#dc2626", "triangle-down"),
        (neutral_bursts, "neutral burst", "#6b7280", "circle"),
    ]:
        fig.add_trace(go.Scattergl(x=data["ts"], y=data["mid_at_book"], mode="markers", marker=dict(color=color, size=9, opacity=0.9, symbol=symbol), name=name), row=1, col=1)

    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["signed_share_ma"], mode="lines", line=dict(color="#1d4ed8", width=1.0), opacity=0.35, name=f"signed_share MA ({SIGNED_SHARE_SMOOTH_WINDOW})", hoverinfo="skip"), row=2, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["signed_share"], mode="lines", line=dict(color="#2563eb", width=1.5), opacity=1.0, name="signed_share (raw)"), row=2, col=1)
    fig.add_hline(y=imbalance_threshold, line_dash="dash", line_color="#16a34a", row=2, col=1)
    fig.add_hline(y=-imbalance_threshold, line_dash="dash", line_color="#dc2626", row=2, col=1)

    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["volume_imbalance_ma"], mode="lines", line=dict(color="#7c3aed", width=1.0), opacity=0.35, name=f"volume_imbalance MA ({VOLUME_IMBALANCE_SMOOTH_WINDOW})", hoverinfo="skip"), row=3, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["volume_imbalance"], mode="lines", line=dict(color="#8b5cf6", width=1.5), opacity=1.0, name="volume_imbalance (raw)"), row=3, col=1)
    fig.add_hline(y=imbalance_threshold, line_dash="dash", line_color="#7c3aed", row=3, col=1)
    fig.add_hline(y=-imbalance_threshold, line_dash="dash", line_color="#a78bfa", row=3, col=1)

    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["trades_per_sec_current"], mode="lines", line=dict(color="#ca8a04", width=1.2), name="current trades/sec"), row=4, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["baseline_trades_per_sec"], mode="lines", line=dict(color="#78716c", width=1, dash="dash"), name="baseline trades/sec"), row=4, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["trades_per_sec_to_baseline"], mode="lines", line=dict(color="#b45309", width=1.2), name="trades/sec to baseline"), row=5, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["volume_to_baseline"], mode="lines", line=dict(color="#7c3aed", width=1.2), name="volume to baseline"), row=6, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["volume_current"], mode="lines", line=dict(color="#0f766e", width=1.2), name="current volume"), row=7, col=1)

    hover_x = plot_df["ts"].iloc[0]
    for row in range(1, 8):
        yref = "y domain" if row == 1 else f"y{row} domain"
        fig.add_shape(type="line", xref="x", yref=yref, x0=hover_x, x1=hover_x, y0=0, y1=1, line=dict(color="#6b7280", width=1, dash="dash"), visible=False, layer="above")

    fig.update_layout(height=1580, hovermode="x", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0), margin=dict(l=70, r=30, t=90, b=50))
    fig.update_yaxes(title_text="mid", row=1, col=1)
    fig.update_yaxes(title_text="signed share", row=2, col=1)
    fig.update_yaxes(title_text="volume imbalance", row=3, col=1)
    fig.update_yaxes(title_text="trades / sec", row=4, col=1)
    fig.update_yaxes(title_text="rate ratio", row=5, col=1)
    fig.update_yaxes(title_text="vol ratio", row=6, col=1)
    fig.update_yaxes(title_text="volume", row=7, col=1)
    fig.update_xaxes(title_text="UTC time", row=7, col=1)
    fig._guide_count = 7
    fig._plot_title = title
    return fig

\
def render_shared_hover_figure(fig: go.Figure) -> None:
    wrapper_id = f"shared-hover-{uuid.uuid4().hex}"
    guide_count = int(getattr(fig, "_guide_count", 1))
    graph_html = fig.to_html(
        full_html=False,
        include_plotlyjs=True,
        config={"responsive": True, "scrollZoom": True, "displaylogo": False},
    )
    plot_title = getattr(fig, "_plot_title", "")
    html = ('''
<div id="__WRAPPER__" style="margin-bottom: 1rem;">
  <div style="font-size: 18px; font-weight: 600; margin: 0 0 0.5rem 0;">__PLOT_TITLE__</div>
  __GRAPH__
</div>
<script>
(function() {
  const root = document.getElementById(__WRAPPER_JSON__);
  if (!root) {
    return;
  }

  function attach() {
    const graph = root.querySelector('.plotly-graph-div');
    if (!graph || typeof graph.on !== 'function') {
      setTimeout(attach, 50);
      return;
    }

    const guideCount = __GUIDE_COUNT__;
    const shapeStart = Array.isArray(graph.layout.shapes) ? graph.layout.shapes.length - guideCount : -1;
    if (shapeStart < 0) {
      return;
    }
    const shapeIndices = Array.from({ length: guideCount }, (_, idx) => shapeStart + idx);

    function setLine(xValue, visible) {
      const update = {};
      shapeIndices.forEach((shapeIndex) => {
        update[`shapes[${shapeIndex}].visible`] = visible;
        if (xValue !== undefined && xValue !== null) {
          update[`shapes[${shapeIndex}].x0`] = xValue;
          update[`shapes[${shapeIndex}].x1`] = xValue;
        }
      });
      Plotly.relayout(graph, update);
    }

    graph.on('plotly_hover', function(evt) {
      const point = evt && evt.points && evt.points[0];
      if (!point) {
        return;
      }
      setLine(point.x, true);
    });

    graph.on('plotly_unhover', function() {
      setLine(null, false);
    });
  }

  attach();
})();
</script>
'''
    .replace("__WRAPPER__", wrapper_id)
    .replace("__WRAPPER_JSON__", json.dumps(wrapper_id))
    .replace("__GRAPH__", graph_html)
    .replace("__PLOT_TITLE__", plot_title)
    .replace("__GUIDE_COUNT__", str(guide_count))
    )
    display(HTML(html))
\


def draw_burst_plot(
    burst_minutes: float,
    calm_minutes: float,
    burst_mode: str,
    ratio_threshold: float,
    imbalance_threshold: float,
    require_imbalance: bool,
    cooldown_minutes: float,
):
    features = build_clock_burst_features(
        trade_frame,
        burst_minutes=float(burst_minutes),
        calm_minutes=float(calm_minutes),
    )
    features = add_burst_flags(
        features,
        burst_mode=str(burst_mode),
        ratio_threshold=float(ratio_threshold),
        imbalance_threshold=float(imbalance_threshold),
        require_imbalance=bool(require_imbalance),
        cooldown_minutes=float(cooldown_minutes),
    )
    summary = pd.Series({
        "burst_minutes": float(burst_minutes),
        "calm_minutes": float(calm_minutes),
        "burst_mode": burst_mode,
        "ratio_threshold": float(ratio_threshold),
        "imbalance_threshold": float(imbalance_threshold),
        "require_imbalance": bool(require_imbalance),
        "cooldown_minutes": float(cooldown_minutes),
        "rows": len(features),
        "raw_burst_windows": int(features["raw_burst"].sum()),
        "cooldown_burst_markers": int(features["burst_marker"].sum()),
        "buy_burst_markers": int(((features["burst_marker"]) & (features["burst_side"] > 0)).sum()),
        "sell_burst_markers": int(((features["burst_marker"]) & (features["burst_side"] < 0)).sum()),
        "neutral_burst_markers": int(((features["burst_marker"]) & (features["burst_side"] == 0)).sum()),
    })
    display(summary.to_frame("value"))
    fig = make_burst_figure(
        features,
        imbalance_threshold=float(imbalance_threshold),
        title=f"{SYMBOL} {REFERENCE_DAY}: calm-to-burst activity over {float(burst_minutes):g} min windows",
    )
    render_shared_hover_figure(fig)
    global latest_burst_features, latest_burst_config
    latest_burst_features = features
    latest_burst_config = summary
    return features


BURST_MIN_WIDGET = widgets.FloatSlider(value=DEFAULT_BURST_MINUTES, min=0.5, max=30.0, step=0.5, description="burst min", continuous_update=False)
CALM_MIN_WIDGET = widgets.FloatSlider(value=DEFAULT_CALM_MINUTES, min=5.0, max=120.0, step=5.0, description="calm min", continuous_update=False)
BURST_MODE_WIDGET = widgets.Dropdown(options=["trades/sec", "volume", "both"], value="trades/sec", description="burst mode")
RATIO_THRESHOLD_WIDGET = widgets.FloatSlider(value=DEFAULT_RATIO_THRESHOLD, min=1.0, max=10.0, step=0.25, description="ratio th", continuous_update=False)
IMBALANCE_THRESHOLD_WIDGET = widgets.FloatSlider(value=DEFAULT_IMBALANCE_THRESHOLD, min=0.10, max=1.00, step=0.05, description="imb th", continuous_update=False)
REQUIRE_IMBALANCE_WIDGET = widgets.Checkbox(value=True, description="require imbalance")
COOLDOWN_WIDGET = widgets.FloatSlider(value=DEFAULT_COOLDOWN_MINUTES, min=0.0, max=30.0, step=0.5, description="cooldown min", continuous_update=False)
REFRESH_BUTTON = widgets.Button(description="Refresh plot", button_style="primary")
REFRESH_OUTPUT = widgets.Output()


def _refresh_burst_plot(_=None):
    with REFRESH_OUTPUT:
        REFRESH_OUTPUT.clear_output(wait=True)
        draw_burst_plot(
            burst_minutes=BURST_MIN_WIDGET.value,
            calm_minutes=CALM_MIN_WIDGET.value,
            burst_mode=BURST_MODE_WIDGET.value,
            ratio_threshold=RATIO_THRESHOLD_WIDGET.value,
            imbalance_threshold=IMBALANCE_THRESHOLD_WIDGET.value,
            require_imbalance=REQUIRE_IMBALANCE_WIDGET.value,
            cooldown_minutes=COOLDOWN_WIDGET.value,
        )


def _observe_widget_change(change):
    if change.get("name") == "value":
        _refresh_burst_plot()


REFRESH_BUTTON.on_click(_refresh_burst_plot)
for widget in [BURST_MIN_WIDGET, CALM_MIN_WIDGET, BURST_MODE_WIDGET, RATIO_THRESHOLD_WIDGET, IMBALANCE_THRESHOLD_WIDGET, REQUIRE_IMBALANCE_WIDGET, COOLDOWN_WIDGET]:
    widget.observe(_observe_widget_change, names="value")

display(widgets.VBox([
    widgets.HBox([BURST_MIN_WIDGET, CALM_MIN_WIDGET]),
    widgets.HBox([BURST_MODE_WIDGET, RATIO_THRESHOLD_WIDGET, IMBALANCE_THRESHOLD_WIDGET]),
    widgets.HBox([REQUIRE_IMBALANCE_WIDGET, COOLDOWN_WIDGET]),
    REFRESH_BUTTON,
    REFRESH_OUTPUT,
]))
_refresh_burst_plot()
